In [11]:
import json
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, classification_report
)
from sklearn.model_selection import StratifiedKFold
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
TRAIN_EMB_PATH = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/merged data embeddings/new_data_dense_embeddings.npz"
VAL_EMB_PATH   = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/validation data embeddings/validation_dense_embeddings.npz"
TEST_EMB_PATH  = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/test data embeddings/test_dense_embeddings.npz"
TRAIN_JSON     = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/train.json"
VAL_JSON       = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/validation.json"
TEST_JSON      = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/test.json"
SEMANTIC_MATCHES = r"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/semantic_matches.json"

label_map = {"False": 0, "True": 1, "Conflicting": 2}
reverse_label_map = {0: "False", 1: "True", 2: "Conflicting"}

# ============================================================
# CONFIG
# ============================================================
EMB_DIM   = 1024
TRACE_MAX = 20
SEED      = 42

# ============================================================
# UTILITY
# ============================================================

def load_json(path):
    with open(path) as f:
        return json.load(f)

def load_npz(path):
    return np.load(path)

def numpy_to_python(obj):
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    if isinstance(obj, dict):        return {k: numpy_to_python(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):return [numpy_to_python(i) for i in obj]
    return obj

# ============================================================
# FEATURE ENGINEERING
# ============================================================

def split_and_pad(emb_array, max_traces=TRACE_MAX, emb_dim=EMB_DIM):
    """Row 0 = claim emb. Rows 1: = trace embs."""
    claim_emb = emb_array[0].astype(np.float32)
    traces    = emb_array[1:].astype(np.float32)
    n = traces.shape[0]
    if n < max_traces:
        pad    = np.zeros((max_traces - n, emb_dim), dtype=np.float32)
        traces = np.vstack([traces, pad])
    else:
        traces = traces[:max_traces]
    mask = np.zeros(max_traces, dtype=np.float32)
    mask[:min(n, max_traces)] = 1.0
    return claim_emb, traces, mask


def verdict_features(verdict_list, max_traces=TRACE_MAX):
    """Scalar features from per-trace verdict votes."""
    counts = {0: 0, 1: 0, 2: 0}
    for v in verdict_list:
        if v in label_map:
            counts[label_map[v]] += 1
    total = sum(counts.values())
    if total == 0:
        return np.zeros(7, dtype=np.float32)
    frac_false   = counts[0] / total
    frac_true    = counts[1] / total
    frac_conf    = counts[2] / total
    majority_frac   = max(counts.values()) / total
    distinct_ratio  = sum(1 for c in counts.values() if c > 0) / 3.0
    fracs = [frac_false, frac_true, frac_conf]
    entropy = -sum(p * np.log(p + 1e-9) for p in fracs if p > 0)
    num_traces_norm = min(total / max_traces, 1.0)
    return np.array([frac_false, frac_true, frac_conf, majority_frac,
                     distinct_ratio, entropy, num_traces_norm], dtype=np.float32)


def build_features(emb_array, verdict_list):
    """
    Feature vector per sample:
      - claim_emb        : 1024-d   (raw claim embedding)
      - attn_pooled      : 1024-d   (softmax-weighted mean of trace embs, weight = cosine sim to claim)
      - trace_mean       : 1024-d   (unweighted mean of valid traces)
      - trace_std        : 1024-d   (std of valid traces — disagreement signal)
      - trace_max        : 1024-d   (element-wise max of valid traces)
      - delta_claim_mean : 1024-d   (claim − trace_mean)
      - scalars          : 7-d      (verdict vote fractions, entropy, etc.)
    Total: 1024×6 + 7 = 6151-d
    """
    claim, traces, mask = split_and_pad(emb_array)
    n_valid = int(mask.sum())
    valid   = traces[:n_valid] if n_valid > 0 else traces[:1]

    # Cosine-similarity attention weights
    claim_norm   = claim / (np.linalg.norm(claim) + 1e-9)
    trace_norms  = valid / (np.linalg.norm(valid, axis=1, keepdims=True) + 1e-9)
    sims         = trace_norms @ claim_norm                      # (n_valid,)
    weights      = np.exp(sims) / (np.exp(sims).sum() + 1e-9)   # softmax
    attn_pooled  = (weights[:, None] * valid).sum(axis=0)

    trace_mean   = valid.mean(axis=0)
    trace_std    = valid.std(axis=0)
    trace_max_f  = valid.max(axis=0)
    delta        = claim - trace_mean
    scalars      = verdict_features(verdict_list)

    return np.concatenate([
        claim, attn_pooled, trace_mean, trace_std, trace_max_f, delta, scalars
    ]).astype(np.float32)


# ============================================================
# ID → ITEM MAPPING
# ============================================================

def create_train_id_to_item():
    semantic_matches = load_json(SEMANTIC_MATCHES)
    train_data = load_json(TRAIN_JSON)
    claim_to_item = {item['claim']: item for item in train_data}
    mapping = {}
    for match in semantic_matches:
        if match['new_claim'] in claim_to_item:
            mapping[match['new_id']] = claim_to_item[match['new_claim']]
    return mapping

def create_val_id_to_item():
    return {idx: item for idx, item in enumerate(load_json(VAL_JSON))}

def create_test_id_to_item():
    return {item['query_id']: item for item in load_json(TEST_JSON)}


# ============================================================
# DATASET BUILDERS
# ============================================================

def build_dataset(emb_data, id_to_item, ids, label_key):
    X, y = [], []
    skipped = 0
    for id_ in ids:
        if id_ not in id_to_item:
            skipped += 1
            continue
        item = id_to_item[id_]
        label_str = item.get(label_key)
        if label_str not in label_map:
            skipped += 1
            continue
        key = f'id_{id_}'
        if key not in emb_data:
            skipped += 1
            continue
        verdict_list = item.get('Verdict_list', [])
        feat = build_features(emb_data[key], verdict_list)
        X.append(feat)
        y.append(label_map[label_str])
    if skipped:
        print(f"  Skipped {skipped} samples (missing emb or label)")
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


def load_all():
    print("Loading TRAIN ...")
    train_emb  = load_npz(TRAIN_EMB_PATH)
    id_to_item = create_train_id_to_item()
    all_ids    = sorted(int(k.split('_')[1]) for k in train_emb.keys() if k.startswith('id_'))
    train_ids  = [i for i in all_ids if i in id_to_item]
    X_tr, y_tr = build_dataset(train_emb, id_to_item, train_ids, 'label')
    print(f"  TRAIN: {X_tr.shape}  labels: {dict(Counter(y_tr.tolist()))}")

    print("Loading VAL ...")
    val_emb    = load_npz(VAL_EMB_PATH)
    val_items  = create_val_id_to_item()
    X_va, y_va = build_dataset(val_emb, val_items, sorted(val_items.keys()), 'label')
    print(f"  VAL:   {X_va.shape}  labels: {dict(Counter(y_va.tolist()))}")

    print("Loading TEST ...")
    test_emb   = load_npz(TEST_EMB_PATH)
    test_items = create_test_id_to_item()
    X_te, y_te = build_dataset(test_emb, test_items, sorted(test_items.keys()), 'Label')
    print(f"  TEST:  {X_te.shape}  labels: {dict(Counter(y_te.tolist()))}")

    return X_tr, y_tr, X_va, y_va, X_te, y_te


# ============================================================
# CLASS-WEIGHT HELPER
# ============================================================

def class_weights(y):
    """Per-class weights: total / (n_classes * count_c)."""
    counts = Counter(y.tolist())
    total  = len(y)
    return {c: total / (3 * counts[c]) for c in range(3)}


def sample_weights(y):
    cw = class_weights(y)
    return np.array([cw[int(yi)] for yi in y], dtype=np.float32)


# ============================================================
# XGBoost PARAMS  (tuned for macro F1 / avg recall)
# ============================================================
# Key levers:
#   scale_pos_weight is per-class in multiclass via sample_weight
#   min_child_weight low → finer splits on minority class
#   subsample / colsample → diversity, reduces overfitting on majority
#   max_depth moderate → enough capacity without majority dominance

XGB_PARAMS = dict(
    objective         = 'multi:softprob',
    num_class         = 3,
    eval_metric       = ['mlogloss'],
    tree_method       = 'hist',      # set 'hist' if no GPU
    device            = 'cuda',          # set 'cpu' if no GPU
    n_estimators      = 1000,
    learning_rate     = 0.05,
    max_depth         = 7,
    min_child_weight  = 2,
    gamma             = 0.1,
    subsample         = 0.8,
    colsample_bytree  = 0.6,
    colsample_bylevel = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    seed              = SEED,
    verbosity         = 1,
)


# ============================================================
# METRICS REPORT
# ============================================================

def generate_report(y_true, y_pred, set_name="VAL"):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    cm        = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    precision = precision_score(y_true, y_pred, average=None, zero_division=0)
    recall    = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1        = f1_score(y_true, y_pred, average=None, zero_division=0)

    print(f"\n{'='*70}")
    print(f"{set_name} SET — DETAILED METRICS")
    print(f"{'='*70}\n")
    print("Confusion matrix (rows=true, cols=pred, order=False/True/Conflicting):")
    print(cm, "\n")
    print(f"{'Class':<12}{'Label':<16}{'Precision':<12}{'Recall':<12}{'F1':<10}")
    print("-" * 62)
    for c in range(3):
        print(f"{c:<12}{reverse_label_map[c]:<16}{precision[c]:<12.4f}{recall[c]:<12.4f}{f1[c]:<10.4f}")

    macro_p  = float(np.mean(precision))
    macro_r  = float(np.mean(recall))
    macro_f1 = float(np.mean(f1))
    print("-" * 62)
    print(f"{'MACRO AVG':<28}{macro_p:<12.4f}{macro_r:<12.4f}{macro_f1:<10.4f}\n")

    wp = float(precision_score(y_true, y_pred, average='weighted', zero_division=0))
    wr = float(recall_score(y_true, y_pred, average='weighted', zero_division=0))
    wf = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))
    print(f"Weighted Precision : {wp:.4f}")
    print(f"Weighted Recall    : {wr:.4f}")
    print(f"Weighted F1        : {wf:.4f}\n")

    # Confusion matrix plot
    try:
        plt.figure(figsize=(7, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=list(reverse_label_map.values()),
                    yticklabels=list(reverse_label_map.values()))
        plt.title(f'{set_name} — Confusion Matrix')
        plt.ylabel('True'); plt.xlabel('Predicted')
        plt.tight_layout()
        out = f'/kaggle/working/{set_name.lower()}_cm.png'
        plt.savefig(out, dpi=100, bbox_inches='tight')
        plt.close()
        print(f"Saved: {out}")
    except Exception as e:
        print(f"Plot failed: {e}")

    return dict(
        cm=cm.tolist(), precision=numpy_to_python(precision),
        recall=numpy_to_python(recall), f1=numpy_to_python(f1),
        macro_precision=macro_p, macro_recall=macro_r, macro_f1=macro_f1,
        weighted_precision=wp, weighted_recall=wr, weighted_f1=wf,
    )


# ============================================================
# THRESHOLD SEARCH  (post-hoc, optimises macro F1 on val set)
# ============================================================

def tune_thresholds(proba, y_true, n_steps=30):
    """
    Grid-search per-class threshold offsets (additive to log-proba).
    Returns the offset vector that maximises macro F1 on the given set.
    This compensates for systematic under-prediction of minority classes.
    """
    best_f1   = -1
    best_bias = np.zeros(3)

    # Search biases for minority classes (True=1, Conflicting=2) only;
    # keep False=0 as anchor.
    for b1 in np.linspace(-1.5, 1.5, n_steps):
        for b2 in np.linspace(-1.5, 1.5, n_steps):
            bias    = np.array([0.0, b1, b2])
            biased  = proba + bias[None, :]
            preds   = biased.argmax(axis=1)
            mf1     = f1_score(y_true, preds, average='macro', zero_division=0)
            if mf1 > best_f1:
                best_f1   = mf1
                best_bias = bias.copy()

    print(f"  Best threshold bias: {best_bias}  →  macro F1 = {best_f1:.4f}")
    return best_bias


# ============================================================
# MAIN
# ============================================================

def main():
    X_tr, y_tr, X_va, y_va, X_te, y_te = load_all()

    sw_tr = sample_weights(y_tr)   # up-weight minority classes
    sw_va = sample_weights(y_va)

    # ---------------------------------------------------------
    # DMatrix
    # ---------------------------------------------------------
    dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=sw_tr)
    dval   = xgb.DMatrix(X_va, label=y_va, weight=sw_va)
    dtest  = xgb.DMatrix(X_te, label=y_te)

    # ---------------------------------------------------------
    # Train with early stopping on val mlogloss
    # ---------------------------------------------------------
    print("\nTraining XGBoost ...")
    params_core = {k: v for k, v in XGB_PARAMS.items()
                   if k not in ('n_estimators',)}   # n_estimators → num_boost_round
    evals_result = {}
    # model = xgb.train(
    #     params_core,
    #     dtrain,
    #     num_boost_round    = XGB_PARAMS['n_estimators'],
    #     evals              = [(dtrain, 'train'), (dval, 'val')],
    #     early_stopping_rounds = 40,
    #     evals_result       = evals_result,
    #     verbose_eval       = 50,
    # )
    model = xgb.train(
        params_core,
        dtrain,
        num_boost_round=1000,
        evals=[(dval, 'val')],
        custom_metric=macro_f1_metric,
        maximize=True,
        early_stopping_rounds=40
    )

    best_round = model.best_iteration
    print(f"\nBest round: {best_round}")

    # ---------------------------------------------------------
    # Raw predictions (probabilities)
    # ---------------------------------------------------------
    proba_tr = model.predict(dtrain).reshape(-1, 3)
    proba_va = model.predict(dval).reshape(-1, 3)
    proba_te = model.predict(dtest).reshape(-1, 3)

    # ---------------------------------------------------------
    # Threshold tuning on VAL to boost macro F1 / recall
    # ---------------------------------------------------------
    print("\nTuning class thresholds on VAL set ...")
    bias = tune_thresholds(proba_va, y_va, n_steps=35)

    def apply_bias(proba):
        return (proba + bias[None, :]).argmax(axis=1)

    preds_tr = apply_bias(proba_tr)
    preds_va = apply_bias(proba_va)
    preds_te = apply_bias(proba_te)

    # ---------------------------------------------------------
    # Reports
    # ---------------------------------------------------------
    print("\n" + "="*70 + "\nFINAL EVALUATION\n" + "="*70)
    metrics_tr = generate_report(y_tr, preds_tr, "TRAIN")
    metrics_va = generate_report(y_va, preds_va, "VAL")
    metrics_te = generate_report(y_te, preds_te, "TEST")

    # ---------------------------------------------------------
    # Feature importance plot (top 30)
    # ---------------------------------------------------------
    try:
        feat_imp = model.get_score(importance_type='gain')
        top      = sorted(feat_imp.items(), key=lambda x: -x[1])[:30]
        labels, vals = zip(*top)
        plt.figure(figsize=(10, 7))
        plt.barh(range(len(labels)), vals[::-1])
        plt.yticks(range(len(labels)), labels[::-1], fontsize=9)
        plt.title('Top 30 features by gain')
        plt.tight_layout()
        plt.savefig('/kaggle/working/feature_importance.png', dpi=100, bbox_inches='tight')
        plt.close()
        print("Saved: /kaggle/working/feature_importance.png")
    except Exception as e:
        print(f"Feature importance plot failed: {e}")

    # ---------------------------------------------------------
    # Learning curve
    # ---------------------------------------------------------
    try:
        plt.figure(figsize=(10, 4))
        plt.plot(evals_result['train']['mlogloss'], label='train')
        plt.plot(evals_result['val']['mlogloss'],   label='val')
        plt.axvline(best_round, color='r', linestyle='--', label=f'best ({best_round})')
        plt.xlabel('Round'); plt.ylabel('mlogloss')
        plt.legend(); plt.tight_layout()
        plt.savefig('/kaggle/working/learning_curve.png', dpi=100, bbox_inches='tight')
        plt.close()
        print("Saved: /kaggle/working/learning_curve.png")
    except Exception as e:
        print(f"Learning curve plot failed: {e}")

    # ---------------------------------------------------------
    # Save model + results
    # ---------------------------------------------------------
    model.save_model('/kaggle/working/xgb_model.json')
    print("Model saved: /kaggle/working/xgb_model.json")

    results = dict(
        train_metrics=metrics_tr, val_metrics=metrics_va, test_metrics=metrics_te,
        best_round=best_round,
        threshold_bias=bias.tolist(),
        feature_dim=X_tr.shape[1],
        xgb_params=XGB_PARAMS,
    )
    with open('/kaggle/working/xgb_results.json', 'w') as f:
        json.dump(numpy_to_python(results), f, indent=2)
    print("Results saved: /kaggle/working/xgb_results.json")


if __name__ == '__main__':
    main()

Loading TRAIN ...
  TRAIN: (6400, 6151)  labels: {2: 1508, 0: 3717, 1: 1175}
Loading VAL ...
  VAL:   (1600, 6151)  labels: {2: 383, 0: 913, 1: 304}
Loading TEST ...
  TEST:  (2558, 6151)  labels: {0: 1783, 2: 255, 1: 520}

Training XGBoost ...
[0]	val-mlogloss:1.09355	val-macro_f1:0.40456
[1]	val-mlogloss:1.08889	val-macro_f1:0.38886
[2]	val-mlogloss:1.08022	val-macro_f1:0.43011
[3]	val-mlogloss:1.07784	val-macro_f1:0.40552
[4]	val-mlogloss:1.07643	val-macro_f1:0.38580
[5]	val-mlogloss:1.06840	val-macro_f1:0.40432
[6]	val-mlogloss:1.06672	val-macro_f1:0.38597
[7]	val-mlogloss:1.06078	val-macro_f1:0.41071
[8]	val-mlogloss:1.05451	val-macro_f1:0.42511
[9]	val-mlogloss:1.05317	val-macro_f1:0.40513
[10]	val-mlogloss:1.04756	val-macro_f1:0.41383
[11]	val-mlogloss:1.04110	val-macro_f1:0.43798
[12]	val-mlogloss:1.03558	val-macro_f1:0.44552
[13]	val-mlogloss:1.03058	val-macro_f1:0.46275
[14]	val-mlogloss:1.02650	val-macro_f1:0.46566
[15]	val-mlogloss:1.02092	val-macro_f1:0.48146
[16]	val-mlog

In [12]:
print(0)

0


In [10]:
from sklearn.metrics import f1_score

def macro_f1_metric(predt, dtrain):
    y_true = dtrain.get_label().astype(int)

    predt = predt.reshape(-1, 3)
    y_pred = np.argmax(predt, axis=1)

    f1 = f1_score(y_true, y_pred, average='macro')
    return 'macro_f1', f1